# Multi-Task FM-MLP: Dataset & Model Architecture

In [1]:
import json
import os

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

PROCESSED_DIR = os.path.join(os.getcwd(), "data", "processed")

with open(os.path.join(PROCESSED_DIR, "feature_manifest.json")) as f:
    manifest = json.load(f)

CAT_COLS = [v["column"] for v in manifest["categorical"].values()]
NUM_COLS = manifest["numerical_scaled"] + manifest["numerical_unscaled"]
CARDINALITIES = {v["column"]: v["cardinality"] for v in manifest["categorical"].values()}
EMBED_DIMS = {v["column"]: v["embedding_dim"] for v in manifest["categorical"].values()}

print(f"categorical columns ({len(CAT_COLS)}): {CAT_COLS}")
print(f"numerical columns ({len(NUM_COLS)}): {NUM_COLS}")
print(f"cardinalities: {CARDINALITIES}")
print(f"embedding dims: {EMBED_DIMS}")

categorical columns (4): ['city_enc', 'gender_enc', 'registered_via_enc', 'payment_method_id_enc']
numerical columns (9): ['bd_clean_scaled', 'registration_tenure_days_scaled', 'avg_payment_plan_days_scaled', 'avg_actual_amount_paid_scaled', 'num_transactions_scaled', 'total_secs_log_scaled', 'daily_active_days_scaled', 'is_auto_renew_rate', 'avg_song_completion']
cardinalities: {'city_enc': 22, 'gender_enc': 4, 'registered_via_enc': 7, 'payment_method_id_enc': 40}
embedding dims: {'city_enc': 5, 'gender_enc': 2, 'registered_via_enc': 3, 'payment_method_id_enc': 8}


## PyTorch `Dataset` & `DataLoader`

`CARDINALITIES` already includes the reserved "unknown" bucket from `02_Feature_Engineering.ipynb`'s label encoding (e.g. `city_enc` cardinality is 22, with index 21 reserved for missing/unseen cities), so `nn.Embedding(num_embeddings=cardinality, ...)` is used directly below — no extra `+1` is needed on top of that.

In [2]:
class KKBoxDataset(Dataset):
    def __init__(self, df):
        self.x_cat = torch.tensor(df[CAT_COLS].values, dtype=torch.long)
        self.x_num = torch.tensor(df[NUM_COLS].values, dtype=torch.float32)
        self.y_churn = torch.tensor(df["is_churn"].values, dtype=torch.float32)
        self.y_ltv = torch.tensor(df["log1p_ltv"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.y_churn)

    def __getitem__(self, idx):
        return self.x_num[idx], self.x_cat[idx], self.y_churn[idx], self.y_ltv[idx]


train_df = pd.read_parquet(os.path.join(PROCESSED_DIR, "model_dataset_train.parquet"))
val_df = pd.read_parquet(os.path.join(PROCESSED_DIR, "model_dataset_val.parquet"))
test_df = pd.read_parquet(os.path.join(PROCESSED_DIR, "model_dataset_test.parquet"))

train_ds = KKBoxDataset(train_df)
val_ds = KKBoxDataset(val_df)
test_ds = KKBoxDataset(test_df)

train_loader = DataLoader(train_ds, batch_size=2048, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=2048, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=2048, shuffle=False)

print(f"train: {len(train_ds):,} rows / {len(train_loader)} batches")
print(f"val:   {len(val_ds):,} rows / {len(val_loader)} batches")
print(f"test:  {len(test_ds):,} rows / {len(test_loader)} batches")

train: 1,127,119 rows / 551 batches
val:   241,526 rows / 118 batches
test:  241,526 rows / 118 batches


## 3.4 Factorization Machine Interaction Layer

Rendle, S. (2010), *Factorization Machines*, IEEE ICDM. Computes all pairwise feature interactions in O(k·d) instead of O(d²) via the sum-of-squares-minus-square-of-sums identity.

In [3]:
class FMInteractionLayer(nn.Module):
    def __init__(self, input_dim, k=8):
        super().__init__()
        self.V = nn.Parameter(torch.randn(input_dim, k) * 0.01)

    def forward(self, x):  # x shape: (batch, input_dim)
        xV = x.unsqueeze(2) * self.V.unsqueeze(0)  # (batch, input_dim, k)
        sum_then_sq = xV.sum(dim=1).pow(2)  # (batch, k)
        sq_then_sum = xV.pow(2).sum(dim=1)  # (batch, k)
        return 0.5 * (sum_then_sq - sq_then_sum)  # (batch, k)

## 4.1 Full Model Architecture (FM + MLP + Dual Heads)

Embeddings + numerical features are concatenated (27 dims), passed through the FM layer (8-dim interaction vector), concatenated again (35 dims), then through the shared MLP backbone (Dense+BatchNorm+ReLU+Dropout × 3, ending at a 64-dim shared representation) into two independent heads.

In [4]:
class MultiTaskFMNet(nn.Module):
    def __init__(self, cat_cols, cardinalities, embed_dims, num_numerical, fm_k=8, backbone_dims=(256, 128, 64)):
        super().__init__()
        self.cat_cols = cat_cols
        self.embeddings = nn.ModuleDict(
            {col: nn.Embedding(cardinalities[col], embed_dims[col]) for col in cat_cols}
        )
        combined_dim = sum(embed_dims[c] for c in cat_cols) + num_numerical
        self.fm = FMInteractionLayer(combined_dim, k=fm_k)

        backbone_input = combined_dim + fm_k
        layers = []
        prev = backbone_input
        dropouts = [0.3, 0.3, 0.2]
        for dim, p in zip(backbone_dims, dropouts):
            layers += [nn.Linear(prev, dim), nn.BatchNorm1d(dim), nn.ReLU(), nn.Dropout(p)]
            prev = dim
        self.backbone = nn.Sequential(*layers)

        self.churn_head = nn.Sequential(nn.Linear(prev, 32), nn.ReLU(), nn.Linear(32, 1))
        self.ltv_head = nn.Sequential(nn.Linear(prev, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x_num, x_cat):
        embeds = [self.embeddings[col](x_cat[:, i]) for i, col in enumerate(self.cat_cols)]
        x = torch.cat(embeds + [x_num], dim=1)
        fm_out = self.fm(x)
        h = torch.cat([x, fm_out], dim=1)
        shared = self.backbone(h)
        churn_logit = self.churn_head(shared).squeeze(-1)  # logit; sigmoid applied at inference
        ltv_pred = self.ltv_head(shared).squeeze(-1)  # predicts log1p(LTV) directly
        return churn_logit, ltv_pred

## Instantiate and inspect the architecture

In [5]:
def count_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


model = MultiTaskFMNet(
    cat_cols=CAT_COLS,
    cardinalities=CARDINALITIES,
    embed_dims=EMBED_DIMS,
    num_numerical=len(NUM_COLS),
    fm_k=8,
    backbone_dims=(256, 128, 64),
)
print(model)

combined_dim = sum(EMBED_DIMS.values()) + len(NUM_COLS)
print(f"\ncombined pre-FM input dim: {combined_dim} (plan: 27)")
print(f"backbone input dim (post-FM): {combined_dim + 8} (plan: 35)")

print("\nparameter counts:")
for name, part in [
    ("embeddings", model.embeddings),
    ("fm", model.fm),
    ("backbone", model.backbone),
    ("churn_head", model.churn_head),
    ("ltv_head", model.ltv_head),
]:
    print(f"  {name:12s} {count_params(part):>8,}")
print(f"  {'total':12s} {count_params(model):>8,}")

MultiTaskFMNet(
  (embeddings): ModuleDict(
    (city_enc): Embedding(22, 5)
    (gender_enc): Embedding(4, 2)
    (registered_via_enc): Embedding(7, 3)
    (payment_method_id_enc): Embedding(40, 8)
  )
  (fm): FMInteractionLayer()
  (backbone): Sequential(
    (0): Linear(in_features=35, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=128, out_features=64, bias=True)
    (9): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.2, inplace=False)
  )
  (churn_head): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=Tru

## Forward + backward pass sanity check

Pull one real batch, run it through the model, compute both losses (`BCEWithLogitsLoss` with `pos_weight` derived from the actual training-set class imbalance, and `MSELoss` on `log1p_ltv`), and confirm gradients flow back through the whole network. The loss values themselves are meaningless here since the model is randomly initialized — this only confirms the architecture and loss wiring are correct before any training happens.

In [6]:
x_num, x_cat, y_churn, y_ltv = next(iter(train_loader))
print(f"batch shapes: x_num={tuple(x_num.shape)}, x_cat={tuple(x_cat.shape)}, "
      f"y_churn={tuple(y_churn.shape)}, y_ltv={tuple(y_ltv.shape)}")

model.train()
churn_logit, ltv_pred = model(x_num, x_cat)
print(f"forward output shapes: churn_logit={tuple(churn_logit.shape)}, ltv_pred={tuple(ltv_pred.shape)}")

pos_weight = torch.tensor((train_df["is_churn"] == 0).sum() / (train_df["is_churn"] == 1).sum())
print(f"pos_weight (from actual train churn rate): {pos_weight.item():.2f} (plan estimate: ~12-14)")

bce_loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
mse_loss_fn = nn.MSELoss()
loss_churn = bce_loss_fn(churn_logit, y_churn)
loss_ltv = mse_loss_fn(ltv_pred, y_ltv)
print(f"sanity-check losses (untrained model): bce={loss_churn.item():.4f}, mse={loss_ltv.item():.4f}")

(loss_churn + loss_ltv).backward()
all_grads_populated = all(p.grad is not None for p in model.parameters() if p.requires_grad)
print(f"backward() OK, all parameters have gradients: {all_grads_populated}")

batch shapes: x_num=(2048, 9), x_cat=(2048, 4), y_churn=(2048,), y_ltv=(2048,)
forward output shapes: churn_logit=(2048,), ltv_pred=(2048,)
pos_weight (from actual train churn rate): 0.36 (plan estimate: ~12-14)
sanity-check losses (untrained model): bce=0.3646, mse=16.9441
backward() OK, all parameters have gradients: True
